# APEXLINE — YOLOv8n fine-tune (Google Colab, free T4)

Fine-tunes `yolov8n.pt` (and optionally `yolov8n-seg.pt`) on open racing-car
datasets so recall on open-wheelers/karts beats zero-shot COCO.

**Gated decision:** run zero-shot first on your golden clip. Only fine-tune if
recall is visibly poor. Budget: ≤ 2 hours wall-clock including download.

**Runtime → Change runtime type → T4 GPU** before running anything.


In [ ]:
# 1 · Environment
!nvidia-smi -L
!pip -q install ultralytics roboflow
import ultralytics; ultralytics.checks()

## 2 · Dataset (Roboflow Universe, free)

1. Create a free account at https://universe.roboflow.com (Google sign-in works).
2. Search **"Formula 1"** or **"race car detection"**. Good starting points:
   the F1-car detection sets (~116 imgs) and FormulaTracker (~442 imgs, per-team
   classes — we merge them to one `racecar` class below). Check each dataset's
   license page (most Universe sets are CC BY 4.0 — record it in DISCLOSURES.md).
3. On the dataset page: **Download Dataset → YOLOv8 → show download code** and
   paste your snippet below (it contains your private key — do NOT commit it).

If recall on karts matters, add a karting dataset from Universe the same way
and train on the merged folder.


In [ ]:
# 2 · Paste YOUR Roboflow download snippet here (replace placeholders)
from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_KEY")          # never commit this key
project = rf.workspace("WORKSPACE").project("PROJECT")
dataset = project.version(1).download("yolov8")
DATA_YAML = dataset.location + "/data.yaml"
print(DATA_YAML)

In [ ]:
# 3 · Merge all classes to a single 'racecar' class (robust to per-team labels)
import yaml, glob, os
d = yaml.safe_load(open(DATA_YAML))
n = len(d["names"]) if isinstance(d["names"], list) else len(d["names"].keys())
d["names"] = ["racecar"]; d["nc"] = 1
yaml.safe_dump(d, open(DATA_YAML, "w"))
for split in ("train","valid","test"):
    for f in glob.glob(os.path.join(dataset.location, split, "labels", "*.txt")):
        lines = [("0" + l[l.index(" "):]) if " " in l else l for l in open(f)]
        open(f, "w").writelines(lines)
print("classes merged -> racecar")

In [ ]:
# 4 · Train (detection). ~25-40 min on T4 for 50 epochs at this size.
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data=DATA_YAML, epochs=50, imgsz=960, batch=16, patience=15,
            project="apexline", name="ft_det")
metrics = model.val()
print(metrics.box.map50, metrics.box.map)

In [ ]:
# 5 · (Optional) segmentation head for seg-mask footprints — only if the
# dataset has polygon labels. Skip otherwise; bbox-bottom footprint ships fine.
# seg = YOLO("yolov8n-seg.pt")
# seg.train(data=DATA_YAML, epochs=40, imgsz=960, batch=12, project="apexline", name="ft_seg")

In [ ]:
# 6 · Sanity-check on a frame you upload, then download the weights
from google.colab import files
best = "apexline/ft_det/weights/best.pt"
import shutil; shutil.copy(best, "apexline_yolov8n_ft.pt")
files.download("apexline_yolov8n_ft.pt")

## 7 · Back on your laptop

```bash
mkdir -p weights && mv ~/Downloads/apexline_yolov8n_ft.pt weights/
# point config/pipeline.yaml -> model.weights: weights/apexline_yolov8n_ft.pt
python -m pipeline.runner --video data/clips_src/golden.mp4 \
    --calibration data/calibrations/corner_1.json \
    --weights weights/apexline_yolov8n_ft.pt
```

**Fallback if fine-tuning fails or overruns:** ship zero-shot `yolov8n.pt` with
COCO classes {car, truck, motorcycle} merged (already the default in
`pipeline/see/detect.py`). The product demo does not depend on this notebook.

Add every dataset + its license and this notebook to `DISCLOSURES.md`.
